In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import spacy
import networkx as nx

nlp = spacy.load("en_core_web_sm")

# text = """
# Elon Musk founded SpaceX in 2002. SpaceX launched the Falcon 9 rocket from
# Cape Canaveral. NASA awarded SpaceX a $2.6 billion contract for the Artemis
# program. Jim Bridenstine, the former NASA administrator, praised SpaceX for
# reducing launch costs. Boeing also competes for NASA contracts through the
# Starliner program. Jeff Bezos founded Blue Origin, which rivals SpaceX in
# the commercial space industry.
# """

with open("../data/sample.txt", "r") as f:
    text = f.read()

doc = nlp(text)

G = nx.DiGraph()

for ent in doc.ents:
    G.add_node(ent.text, label=ent.label_)

for sent in doc.sents:
    ents_in_sent = list(sent.ents)
    for i in range(len(ents_in_sent)):
        for j in range(i + 1, len(ents_in_sent)):
            e1 = ents_in_sent[i]
            e2 = ents_in_sent[j]
            G.add_edge(e1.text, e2.text, relation="co_occurs", sentence=sent.text.strip())

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
for u, v, data in G.edges(data=True):
    print(f"  {u} -> {v}")


Nodes: 8, Edges: 16
  six -> four
  six -> Anthropic Claude
  six -> Qwen
  six -> OpenAI GPT-5.4
  six -> three
  four -> Anthropic Claude
  four -> Qwen
  four -> OpenAI GPT-5.4
  four -> three
  Anthropic Claude -> Qwen
  Anthropic Claude -> OpenAI GPT-5.4
  Anthropic Claude -> three
  Qwen -> OpenAI GPT-5.4
  Qwen -> three
  OpenAI GPT-5.4 -> three
  causal tracing -> linear


In [25]:
# Extracting NER relationships
entities = [(ent.text, ent.label_) for ent in doc.ents]
for text_span, label in entities:
    print(f"{text_span:25s} {label}")

six                       CARDINAL
four                      CARDINAL
Anthropic Claude          PERSON
Qwen                      PERSON
OpenAI GPT-5.4            ORG
three                     CARDINAL
causal tracing            PERSON
linear                    ORG


In [27]:
# Extracting relationship between entities

G = nx.DiGraph()

# Add entity nodes with their NER labels
for ent in doc.ents:
    G.add_node(ent.text, label=ent.label_)

# Add edges for entities co-occurring in the same sentence
for sent in doc.sents:
    ents_in_sent = list(sent.ents)
    for i in range(len(ents_in_sent)):
        for j in range(i + 1, len(ents_in_sent)):
            e1 = ents_in_sent[i]
            e2 = ents_in_sent[j]
            if G.has_edge(e1.text, e2.text):
                # Append sentence to existing edge
                existing = G[e1.text][e2.text]["sentences"]
                existing.append(sent.text.strip())
            else:
                G.add_edge(
                    e1.text,
                    e2.text,
                    relation="co_occurs",
                    sentences=[sent.text.strip()],
                )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph: 8 nodes, 16 edges


In [28]:
def extract_relation(sent, ent1, ent2):
    """Find the root verb between two entities in a sentence."""
    root = sent.root
    if root.pos_ == "VERB":
        return root.lemma_
    # Fall back to any verb in the sentence
    for token in sent:
        if token.pos_ == "VERB":
            return token.lemma_
    return "related_to"

for sent in doc.sents:
    ents_in_sent = list(sent.ents)
    for i in range(len(ents_in_sent)):
        for j in range(i + 1, len(ents_in_sent)):
            rel = extract_relation(sent, ents_in_sent[i], ents_in_sent[j])
            print(f"{ents_in_sent[i].text} --{rel}--> {ents_in_sent[j].text}")


six --related_to--> four
six --related_to--> Anthropic Claude
six --related_to--> Qwen
six --related_to--> OpenAI GPT-5.4
six --related_to--> three
four --related_to--> Anthropic Claude
four --related_to--> Qwen
four --related_to--> OpenAI GPT-5.4
four --related_to--> three
Anthropic Claude --related_to--> Qwen
Anthropic Claude --related_to--> OpenAI GPT-5.4
Anthropic Claude --related_to--> three
Qwen --related_to--> OpenAI GPT-5.4
Qwen --related_to--> three
OpenAI GPT-5.4 --related_to--> three
causal tracing --converge--> linear


In [32]:
# querying the knowledge graph
# Find all neighbors of an entity
qwen_neighbors = list(G.neighbors("Qwen"))
print(f"Qwen connects to: {qwen_neighbors}")

# Find shortest path between two entities
try:
    path = nx.shortest_path(G, source="Qwen", target="OpenAI GPT-5.4")
    print(f"Path from Qwen to OpenAI GPT-5.4: {' -> '.join(path)}")
except nx.NetworkXNoPath:
    print("No path found")

# Get all connected components (clusters of related entities)
undirected = G.to_undirected()
components = list(nx.connected_components(undirected))
for i, comp in enumerate(components):
    print(f"Cluster {i}: {comp}")

# Find the most connected entities (by degree)
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)
print("\nMost connected entities:")
for entity, degree in degree_sorted[:5]:
    print(f"  {entity}: {degree} connections")

# Get all edges for a specific entity with their data
for u, v, data in G.edges("Qwen", data=True):
    print(f"  {u} -> {v}: {data.get('relation', 'unknown')}")


Qwen connects to: ['OpenAI GPT-5.4', 'three']
Path from Qwen to OpenAI GPT-5.4: Qwen -> OpenAI GPT-5.4
Cluster 0: {'OpenAI GPT-5.4', 'six', 'three', 'Qwen', 'four', 'Anthropic Claude'}
Cluster 1: {'causal tracing', 'linear'}

Most connected entities:
  six: 5 connections
  four: 5 connections
  Anthropic Claude: 5 connections
  Qwen: 5 connections
  OpenAI GPT-5.4: 5 connections
  Qwen -> OpenAI GPT-5.4: co_occurs
  Qwen -> three: co_occurs


In [35]:
import spacy

nlp = spacy.load("en_core_web_sm")
print("Pipeline:", nlp.pipe_names)
doc = nlp("I was reading the paper.")
token = doc[1]  # 'I'
print(token.morph)  # 'Case=Nom|Number=Sing|Person=1|PronType=Prs'
print(token.morph.get("Tense"))  # ['Prs']

Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
['Past']


In [38]:
TRAIN_DATA = [('The F15 aircraft uses a lot of fuel', {'entities': [(4, 7, 'aircraft')]}),
 ('did you see the F16 landing?', {'entities': [(16, 19, 'aircraft')]}),
 ('how many missiles can a F35 carry', {'entities': [(24, 27, 'aircraft')]}),
 ('is the F15 outdated', {'entities': [(7, 10, 'aircraft')]}),
 ('does the US still train pilots to dog fight?',
  {'entities': [(0, 0, 'aircraft')]}),
 ('how long does it take to train a F16 pilot',
  {'entities': [(33, 36, 'aircraft')]}),
 ('how much does a F35 cost', {'entities': [(16, 19, 'aircraft')]}),
 ('would it be possible to steal a F15', {'entities': [(32, 35, 'aircraft')]}),
 ('who manufactures the F16', {'entities': [(21, 24, 'aircraft')]}),
 ('how many countries have bought the F35',
  {'entities': [(35, 38, 'aircraft')]}),
 ('is the F35 a waste of money', {'entities': [(7, 10, 'aircraft')]})]

TRAIN_DATA

[('The F15 aircraft uses a lot of fuel', {'entities': [(4, 7, 'aircraft')]}),
 ('did you see the F16 landing?', {'entities': [(16, 19, 'aircraft')]}),
 ('how many missiles can a F35 carry', {'entities': [(24, 27, 'aircraft')]}),
 ('is the F15 outdated', {'entities': [(7, 10, 'aircraft')]}),
 ('does the US still train pilots to dog fight?',
  {'entities': [(0, 0, 'aircraft')]}),
 ('how long does it take to train a F16 pilot',
  {'entities': [(33, 36, 'aircraft')]}),
 ('how much does a F35 cost', {'entities': [(16, 19, 'aircraft')]}),
 ('would it be possible to steal a F15', {'entities': [(32, 35, 'aircraft')]}),
 ('who manufactures the F16', {'entities': [(21, 24, 'aircraft')]}),
 ('how many countries have bought the F35',
  {'entities': [(35, 38, 'aircraft')]}),
 ('is the F35 a waste of money', {'entities': [(7, 10, 'aircraft')]})]

In [ ]:
# visualizing the knowledge graph
#
# import matplotlib.pyplot as plt
# import networkx as nx
#
# # Color nodes by entity type
# color_map = {
#     "PERSON": "#4ade80",
#     "ORG": "#60a5fa",
#     "GPE": "#f97316",
#     "DATE": "#a78bfa",
#     "MONEY": "#facc15",
#     "PRODUCT": "#f472b6",
# }
#
# node_colors = []
# for node in G.nodes():
#     label = G.nodes[node].get("label", "")
#     node_colors.append(color_map.get(label, "#9ca3af"))
#
# plt.figure(figsize=(14, 10))
# pos = nx.spring_layout(G, seed=42, k=2)
#
# nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, alpha=0.9)
# nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold")
# nx.draw_networkx_edges(G, pos, edge_color="#555555", arrows=True, arrowsize=15, alpha=0.6)
#
# plt.title("Knowledge Graph: Entities and Relationships")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig("knowledge_graph.png", dpi=150, bbox_inches="tight")
# plt.show()


In [10]:
import scrapetube

videos = scrapetube.get_channel(channel_id='UCuyRsHZILrU7ZDIAbGASHdA',  limit=1, sleep=1)

for video in videos:
    print(video['videoId'])

KeyboardInterrupt: 

In [24]:
import yt_dlp

channel_url = "https://www.youtube.com/@ThePrintIndia/videos"

options = {
    "extract_flat": "in_playlist",
    "skip_download": True,
    "quiet": True,
}

video_details = []

with yt_dlp.YoutubeDL(options) as ydl:
    result = ydl.extract_info(channel_url, download=False)

    for video in result.get("entries", []):
        if video:
            # print(video.get("id"), video.get("title"))
            video_details.append({
                "id": video.get("id"),
                "title": video.get("title"),
                "url": f"https://www.youtube.com/watch?v={video.get('id')}"
            })

In [25]:
video_details = []

for video_entry in result.get("entries", []):
    if video_entry:
        video_details.append({
            "id": video_entry.get("id"),
            "title": video_entry.get("title"),
            "url": f"https://www.youtube.com/watch?v={video_entry.get('id')}",
            "duration": video_entry.get("duration"),
            "view_count": video_entry.get("view_count")
        })

In [34]:
channel_url = "https://www.youtube.com/@ThePrintIndia/podcasts"

options = {
    "extract_flat": "in_playlist",
    "skip_download": True,
    "quiet": True,
}

podcast_details = []

with yt_dlp.YoutubeDL(options) as ydl:
    podcast_result = ydl.extract_info(channel_url, download=False)

    for video in podcast_result.get("entries", []):
        if video:
            # print(video.get("id"), video.get("title"))
            podcast_details.append({
                "id": video.get("id"),
                "title": video.get("title"),
                "url": video.get("url", None),
                "duration": video.get("duration"),
                "view_count": video.get("view_count")
            })

In [37]:
podcast_details[0]

{'id': 'PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y',
 'title': 'Cut The Clutter with Shekhar Gupta',
 'url': 'https://www.youtube.com/playlist?list=PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y',
 'duration': None,
 'view_count': None}

In [38]:
channel_url = "https://www.youtube.com/playlist?list=PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y"

options = {
    "extract_flat": "in_playlist",
    "skip_download": True,
    "quiet": True,
}

cc_details = []

with yt_dlp.YoutubeDL(options) as ydl:
    cc_result = ydl.extract_info(channel_url, download=False)

    for video in cc_result.get("entries", []):
        if video:
            # print(video.get("id"), video.get("title"))
            cc_details.append({
                "id": video.get("id"),
                "title": video.get("title"),
                "url": f"https://www.youtube.com/watch?v={video.get('id')}",
                "duration": video.get("duration"),
                "view_count": video.get("view_count")
            })

In [39]:
cc_result

{'id': 'PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y',
 'title': 'Cut The Clutter with Shekhar Gupta',
 'availability': 'public',
 'channel_follower_count': None,
 'description': 'Cut The Clutter with Shekhar Gupta, a daily decluttering of the top stories of the day.',
 'tags': [],
 'thumbnails': [{'url': 'https://i.ytimg.com/pl_c/PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y/studio_square_thumbnail.jpg?sqp=CNnzpdYG-oaymwEICPABEPABSFqi85f_AwYIzu6DzAY=&rs=AOn4CLDNxxB4Jw_KePADUkgsPZbOD3hiZg',
   'height': 240,
   'width': 240,
   'id': '0',
   'resolution': '240x240'},
  {'url': 'https://i.ytimg.com/pl_c/PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y/studio_square_thumbnail.jpg?sqp=CJi8pdYG-oaymwEICOADEOADSFqi85f_AwYIzu6DzAY=&rs=AOn4CLCAW8h6kAT_-qEd2BpddkFTiCnVCQ',
   'height': 480,
   'width': 480,
   'id': '1',
   'resolution': '480x480'},
  {'url': 'https://i.ytimg.com/pl_c/PLfa0eRI2uIaR1iOJnq7tklxIJDLxTxI5y/studio_square_thumbnail.jpg?sqp=CJLmpdYG-oaymwEICNAFENAFSFqi85f_AwYIzu6DzAY=&rs=AOn4CLBPm8aWHLTzF8hR8EhzE090fi